> **Riverside's constraint:** IT approved one AWS A10G (24 GB VRAM) for a 3-day fine-tuning sprint. The team has three candidate models: GPT-2-Medium (355M params), LLaMA-3-1B, and LLaMA-3-8B. Before writing a single line of training code, they need to know: which models fit in 24 GB, at what precision, and what tricks are needed for the larger ones? This notebook answers with measured numbers, not rules of thumb.

# Mixed Precision and Memory Math: What Fits in Your GPU?

| Part | Concept | Riverside question |
|------|---------|-------------------|
| 1 | Memory footprint math | How many GB does each model need? |
| 2 | fp32 vs fp16 vs bf16 | Will fp16 training save us? What's the risk? |
| 3 | torch.autocast + GradScaler | How do we train stably in fp16? |
| 4 | Gradient checkpointing | Can we trade compute for memory? |
| 5 | Memory profiling | Where exactly does the memory go? |
| 6 | Toy → real bridge | Which Riverside models fit, and how? |

---

> **Prerequisites:** Ch1 GPU Hardware (memory hierarchy, bandwidth concepts).  
> **Connects to:** `learning/genai/04-llm/02-llm-finetuning-parameter-techniques.ipynb` (LoRA memory savings).

In [ ]:
import subprocess, sys
for pkg in ['torch','numpy','matplotlib','transformers']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_GPU = torch.cuda.is_available()

print(f"Device: {DEVICE}")
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  |  VRAM: {props.total_memory/1e9:.1f} GB")
    print(f"  (A10G reference: 24 GB VRAM)")
else:
    print("No GPU — using CPU. Memory calculations shown as reference values.")
    print("All code runs; GPU-specific profiling shows reference numbers.")

A10G_VRAM_GB = 24.0  # Riverside's constraint
print(f"\nRiverside constraint: {A10G_VRAM_GB} GB VRAM on A10G")

---

## Part 1 — Memory Footprint Math: Parameters × Bytes

A model's memory requirement has four components:

| Component | Formula | Notes |
|---|---|---|
| **Parameters** | `P × bytes_per_param` | The model weights |
| **Gradients** | Same as parameters | One gradient per weight |
| **Optimizer states** | 2× parameters (Adam) | m and v per parameter |
| **Activations** | `B × S × H × L × bytes` | Grows with batch size and sequence length |

**Total for full fine-tuning (fp32):** ≈ 4 × parameter memory (params + grads + 2× optimizer)

**For bf16 training:** params + grads in bf16 (2 bytes), optimizer states in fp32 (4 bytes) = 2× params for params/grads + 2× params for optimizer = 4× params total (same as fp32 actually, but with smaller activation memory)

> **Intuition first:** Think of each parameter as one number stored on the GPU — like a single cell in a spreadsheet. fp32 gives each cell 4 bytes; bf16 gives it 2 bytes. A 1B-parameter model in fp32 occupies 4 GB at rest. Training multiplies that: you also need to store the gradients (one per weight, same size), the optimizer's running momentum averages (two per weight for Adam, both in fp32), and the intermediate values from every layer computed during the forward pass. The table below shows each component for the three Riverside candidate models.

![GPT-2-Medium training memory: parameters + gradients + optimizer states + activations stacked, fp32 vs bf16 comparison with A10G 24 GB limit line](images/memory-footprint-breakdown.png)

In [ ]:
# ── Part 1: Memory footprint calculator ──────────────────────────────────────
def memory_breakdown_gb(n_params, precision='fp32', batch=8, seq=512, hidden=1024, layers=12):
    """Calculate GPU memory requirements for fine-tuning."""
    bytes_per_param = {'fp32': 4, 'fp16': 2, 'bf16': 2, 'int8': 1}[precision]

    params_gb     = n_params * bytes_per_param / 1e9
    gradients_gb  = n_params * bytes_per_param / 1e9  # same dtype as params
    # Adam optimizer: m and v in fp32 regardless of training precision
    optimizer_gb  = n_params * 4 * 2 / 1e9           # always fp32
    # Activations: rough estimate (varies widely by architecture)
    activation_gb = batch * seq * hidden * layers * bytes_per_param / 1e9

    total = params_gb + gradients_gb + optimizer_gb + activation_gb
    return {'params': params_gb, 'grads': gradients_gb,
            'optimizer': optimizer_gb, 'activations': activation_gb, 'total': total}

# Model configs
models = {
    'GPT-2-Medium':  {'params': 355e6,  'hidden': 1024, 'layers': 24},
    'LLaMA-3-1B':    {'params': 1e9,    'hidden': 2048, 'layers': 22},
    'LLaMA-3-8B':    {'params': 8e9,    'hidden': 4096, 'layers': 32},
}

print(f"Memory requirements (full fine-tuning, batch=8, seq=512):")
print(f"{'Model':18s}  {'Precision':8s}  {'Params':7s}  {'Grads':7s}  {'Optim':7s}  {'Activ':7s}  {'Total':7s}  {'Fits 24GB?':10s}")
print("-" * 90)
for name, cfg in models.items():
    for prec in ['fp32', 'bf16']:
        mb = memory_breakdown_gb(cfg['params'], prec, hidden=cfg['hidden'], layers=cfg['layers'])
        fits = "\u2713" if mb['total'] < A10G_VRAM_GB else "\u2717"
        print(f"  {name:16s}  {prec:8s}  {mb['params']:5.1f}GB  {mb['grads']:5.1f}GB  "
              f"{mb['optimizer']:5.1f}GB  {mb['activations']:5.1f}GB  {mb['total']:5.1f}GB  {fits}")

---

## Part 2 — fp32, fp16, bf16: Precision Formats

| Format | Bits | Exponent bits | Mantissa bits | Max value |
|--------|------|---------------|---------------|-----------|
| fp32   | 32   | 8             | 23            | ~3.4×10³⁸ |
| fp16   | 16   | 5             | 10            | 65,504     |
| bf16   | 16   | 8             | 7             | ~3.4×10³⁸ |

**fp16 risk:** gradients during LLM training can exceed 65,504 → overflow → NaN loss.  
**bf16 advantage:** same exponent range as fp32 → no overflow risk. Preferred for training.

#### 🔮 Predict first

We run fp16 training on GPT-2-Medium **without** GradScaler (no gradient scaling). Will the loss:

1. **(a) Train normally** — fp16 is sufficient; overflow is rare in practice
2. **(b) Show NaN immediately** — every gradient overflows
3. **(c) Train for a few steps then diverge** — overflows occasionally, causing instability

![fp32 vs fp16 vs bf16 bit layouts: exponent width determines overflow risk; fp16 overflows at 65,504 where LLM gradients live](images/fp32-fp16-bf16-number-line.png)

In [ ]:
# ── Demonstrating fp16 overflow and underflow directly ───────────────────────
# fp16 max value is 65,504 — surprisingly easy to exceed in LLM training
# fp16 min positive subnormal ≈ 5.96e-8 — values below it flush to zero

print("fp16 numeric limits:")
x_large = torch.tensor(70000.0, dtype=torch.float16)
print(f"  70,000 in fp16:        {x_large}       ← overflows to inf!")

x_small = torch.tensor(1e-8, dtype=torch.float16)
print(f"  1e-8 in fp16:          {x_small}       ← underflows to 0.0!")

x_ok = torch.tensor(0.001, dtype=torch.float16)
print(f"  0.001 in fp16:         {x_ok}       ← fine, within range")

print()
print("LLM gradient issue: if any weight gradient spikes to ~65k during training,")
print("  fp16 records it as inf → the optimizer step produces NaN weights → training crashes.")
print("  bf16 max value ≈ 3.4×10³⁸ (same as fp32) → no overflow risk at typical LR scales.")

In [ ]:
# ── Part 2: fp16 overflow demonstration ──────────────────────────────────────
from transformers import GPT2LMHeadModel, AutoTokenizer

print("Loading GPT-2 (small) for precision experiments...")
try:
    model = GPT2LMHeadModel.from_pretrained('gpt2')
    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    MODEL_LOADED = True
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  GPT-2 loaded: {n_params/1e6:.0f}M parameters")
except Exception as e:
    print(f"  GPT-2 not available: {e}")
    MODEL_LOADED = False

if not MODEL_LOADED:
    print("Using a small toy model for the precision demonstration")
    class ToyLM(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.Sequential(*[nn.Linear(256,256) for _ in range(6)])
        def forward(self, x): return self.layers(x).mean()
    model = ToyLM()

model = model.to(DEVICE)

# Training sample
sample_text = "The Riverside editing assistant needs to understand"
if MODEL_LOADED and hasattr(model, 'generate'):
    tokens = tokenizer(sample_text, return_tensors='pt', max_length=32, truncation=True, padding=True)
    input_ids = tokens['input_ids'].to(DEVICE)
    labels = input_ids.clone()

print()
print("Testing fp16 WITHOUT GradScaler:")
losses_fp16 = []
m16 = model.half() if MODEL_LOADED else model
opt = torch.optim.AdamW(m16.parameters(), lr=5e-4)
for step in range(5):
    opt.zero_grad()
    try:
        if MODEL_LOADED:
            with torch.autocast(device_type='cpu' if not HAS_GPU else 'cuda', dtype=torch.float16):
                out = m16(input_ids, labels=labels)
            loss = out.loss
        else:
            with torch.autocast(device_type='cpu' if not HAS_GPU else 'cuda', dtype=torch.float16):
                loss = m16(torch.randn(4, 256).to(DEVICE))
        loss.backward()
        opt.step()
        losses_fp16.append(loss.item())
        print(f"  step {step}: loss = {loss.item():.4f}")
    except Exception as ex:
        print(f"  step {step}: ERROR — {ex}")
        losses_fp16.append(float('nan'))
        break

has_nan = any(np.isnan(l) for l in losses_fp16)
print()
print(f"→ NaN losses occurred: {has_nan}")
if has_nan:
    print("  Prediction (b) or (c) confirmed — fp16 without GradScaler is unstable")
else:
    print("  This run didn't overflow (small model/short sequence)")
    print("  In practice, large LLMs with fp16 + no GradScaler regularly produce NaN gradients")

---

## Part 3 — torch.autocast + GradScaler: Stable Mixed Precision Training

The two-part recipe for safe fp16 training:
1. **`torch.autocast`** — runs forward pass in fp16; keeps loss computation in fp32
2. **`torch.cuda.amp.GradScaler`** — scales the loss before backward (prevents underflow); unscales before optimizer step

This combination gives fp16's speed advantage without its numerical instability.

> **Intuition:** Your gradients are whispered numbers — so small that fp16 rounds them to zero (as shown above with `0.00005`). GradScaler *shouts* them first: it multiplies the loss by 65,536 before the backward pass, so the gradients are loud enough for fp16 to represent. Before the optimizer step, it divides back by 65,536 — the optimizer never sees inflated values. If any gradient explodes to inf despite scaling, GradScaler skips that batch and lowers the scale factor for next time.

In [ ]:
# ── Part 3: torch.autocast + GradScaler ──────────────────────────────────────
print("Training with torch.autocast + GradScaler (stable mixed precision):")

m_stable = (GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)
            if MODEL_LOADED else ToyLM().to(DEVICE))
opt_stable = torch.optim.AdamW(m_stable.parameters(), lr=5e-4)
scaler = torch.cuda.amp.GradScaler(enabled=HAS_GPU)  # no-op on CPU

losses_stable = []
for step in range(5):
    opt_stable.zero_grad()
    dtype = torch.float16 if HAS_GPU else torch.float32
    device_type = 'cuda' if HAS_GPU else 'cpu'

    with torch.autocast(device_type=device_type, dtype=dtype):
        if MODEL_LOADED:
            out = m_stable(input_ids, labels=labels)
            loss = out.loss
        else:
            loss = m_stable(torch.randn(4, 256).to(DEVICE))

    scaler.scale(loss).backward()  # scaled backward
    scaler.step(opt_stable)         # unscale + step
    scaler.update()                 # update scale factor

    losses_stable.append(loss.item())
    print(f"  step {step}: loss = {loss.item():.4f}  (scale = {scaler.get_scale():.0f})")

has_nan_stable = any(np.isnan(l) for l in losses_stable)
print()
print(f"→ NaN losses with GradScaler: {has_nan_stable}")
print("  GradScaler maintains stability by dynamically adjusting gradient scale.")
print()
print("Memory comparison (estimated):")
gpt2_fp32_gb = 355e6 * 4 / 1e9  # params in fp32
gpt2_bf16_gb = 355e6 * 2 / 1e9  # params in bf16
print(f"  GPT-2 params in fp32: {gpt2_fp32_gb:.2f} GB")
print(f"  GPT-2 params in bf16: {gpt2_bf16_gb:.2f} GB  ({gpt2_fp32_gb/gpt2_bf16_gb:.0f}\u00d7 smaller)")

#### What just happened — and what's missing

`torch.autocast` + `GradScaler` gives the speed of fp16 with the stability of fp32. The scale factor starts high (2^16 = 65536) and decreases when NaN/Inf gradients appear.

**Missing piece:** Even with bf16, the activation memory grows with sequence length and batch size. A 8B model with batch=8, seq=512 needs ~30+ GB just for activations — more than the A10G has. We need a way to trade compute for memory → gradient checkpointing.

---

## Part 4 — Gradient Checkpointing: Trading Compute for Memory

By default, PyTorch stores all intermediate activations during the forward pass (needed for backward). **Gradient checkpointing** discards those activations and recomputes them during backward.

Memory saved: O(L) → O(√L) where L = number of layers. In plain terms: instead of storing all N layer activations simultaneously (one buffer per layer), checkpointing keeps only every √N-th layer's output and recomputes the in-between ones during the backward pass. For a 24-layer model, that means keeping ~5 activation buffers instead of 24 — roughly 55-60% less memory, at the cost of running the forward pass once more (adds ~30% compute overhead).  
Compute cost: ~30–40% extra (the forward pass runs twice).

#### 🔮 Predict first

Gradient checkpointing halves peak memory. Does it also halve training time?

1. **(a) Yes** — less memory means faster transfers
2. **(b) No, it adds ~30% compute overhead** — you re-run the forward pass during backward
3. **(c) It actually speeds things up** — less memory = better GPU cache utilization

![Gradient checkpointing tradeoff: peak memory falls as checkpointing frequency increases, but compute overhead rises](images/gradient-checkpointing-tradeoff.png)

In [ ]:
# ── Part 4: Gradient checkpointing memory savings ────────────────────────────
import torch.utils.checkpoint as cp
import time

class DeepNet(nn.Module):
    """Deep network to make checkpointing effects visible."""
    def __init__(self, use_checkpoint=False):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(512, 512) for _ in range(24)])
        self.use_checkpoint = use_checkpoint

    def forward_one(self, layer, x):
        return torch.relu(layer(x))

    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpoint:
                x = cp.checkpoint(self.forward_one, layer, x, use_reentrant=False)
            else:
                x = torch.relu(layer(x))
        return x.mean()

# Measure memory and time (CPU proxy — GPU results would be more dramatic)
import tracemalloc

results = {}
for use_ckpt in [False, True]:
    model_test = DeepNet(use_checkpoint=use_ckpt).to(DEVICE)
    x = torch.randn(16, 512).to(DEVICE)
    opt_test = torch.optim.SGD(model_test.parameters(), lr=0.01)

    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        for _ in range(3):
            opt_test.zero_grad()
            model_test(x).backward()
            opt_test.step()
        if HAS_GPU: torch.cuda.synchronize()
        t_ms = (time.perf_counter() - t0) / 3 * 1000
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
    else:
        tracemalloc.start()
        t0 = time.perf_counter()
        for _ in range(3):
            opt_test.zero_grad()
            model_test(x).backward()
            opt_test.step()
        t_ms = (time.perf_counter() - t0) / 3 * 1000
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        peak_mb = peak / 1e6

    results[use_ckpt] = {'time_ms': t_ms, 'peak_mb': peak_mb}

no_ckpt = results[False]; with_ckpt = results[True]
print(f"Gradient checkpointing results (24-layer network):")
print(f"  Without checkpointing: {no_ckpt['time_ms']:.1f}ms/step, peak {no_ckpt['peak_mb']:.1f} MB")
print(f"  With checkpointing:    {with_ckpt['time_ms']:.1f}ms/step, peak {with_ckpt['peak_mb']:.1f} MB")

if no_ckpt['peak_mb'] > 0:
    mem_ratio = no_ckpt['peak_mb'] / with_ckpt['peak_mb']
    time_ratio = with_ckpt['time_ms'] / no_ckpt['time_ms']
    print(f"\n  Memory reduction: {mem_ratio:.1f}\u00d7")
    print(f"  Time overhead:    {time_ratio:.1f}\u00d7 (slower)")
    if time_ratio > 1.1:
        print("\n  Prediction (b) confirmed: checkpointing adds compute overhead (~30%)")
    else:
        print("\n  On CPU: overhead is less visible. On GPU, expect 30% overhead with 50% memory savings.")

---

## Part 5 — Memory Profiling: Where Does the Memory Go?

`torch.cuda.max_memory_allocated()` measures peak VRAM usage. For a training step, the peak occurs during the backward pass (all activations stored). For inference, the peak is at the largest attention matrix.

In [ ]:
# ── Part 5: Memory profiling breakdown ───────────────────────────────────────
def profile_memory(model_fn, n_params, label):
    """Profile peak memory for one forward+backward step."""
    if HAS_GPU:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    m = model_fn().to(DEVICE)
    x = torch.randn(4, 64, dtype=torch.float32).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)

    # Forward + backward
    opt.zero_grad()
    loss = m(x).mean()
    loss.backward()
    opt.step()

    if HAS_GPU:
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
        total_vram = torch.cuda.get_device_properties(0).total_memory / 1e6
    else:
        # Estimate from parameter count
        peak_mb = n_params * 16 / 1e6  # rough: 16 bytes per param (params+grads+optimizer)
        total_vram = A10G_VRAM_GB * 1000

    param_mb = sum(p.numel() * 4 for p in m.parameters()) / 1e6
    print(f"  {label}: {param_mb:.0f} MB params → {peak_mb:.0f} MB peak VRAM "
          f"({peak_mb/param_mb:.1f}\u00d7 params)")

print("Memory profiling (forward + backward, batch=4, seq=64):")
profile_memory(lambda: nn.Sequential(*[nn.Linear(512,512) for _ in range(4)]),
               4*512*512, "4-layer Linear")
profile_memory(lambda: nn.Sequential(*[nn.Linear(512,512) for _ in range(8)]),
               8*512*512, "8-layer Linear")
profile_memory(lambda: nn.Sequential(*[nn.Linear(512,512) for _ in range(16)]),
               16*512*512, "16-layer Linear")

print()
print("Rule of thumb for full fine-tuning:")
print("  Peak VRAM \u2248 4\u00d7 parameter memory (params + grads + 2\u00d7 optimizer states)")
print("  Activations add proportionally to batch\u00d7seq\u00d7hidden")

---

## Part 6 — Toy → Real: Riverside's A10G Model Selection

With the tools from Parts 1–5, we can now answer Riverside's original question.

In [ ]:
# ── Part 6: Riverside model selection for A10G ────────────────────────────────
print("Riverside A10G (24 GB) — Model Selection Analysis")
print("=" * 60)
print()

scenarios = [
    ("GPT-2-Medium (355M)", 355e6,  1024, 24, 'fp32', False, False),
    ("GPT-2-Medium (355M)", 355e6,  1024, 24, 'bf16', False, False),
    ("LLaMA-3-1B",          1e9,    2048, 22, 'bf16', False, False),
    ("LLaMA-3-1B + ckpt",   1e9,    2048, 22, 'bf16', True,  False),
    ("LLaMA-3-8B",          8e9,    4096, 32, 'bf16', False, False),
    ("LLaMA-3-8B + LoRA",   8e9,    4096, 32, 'bf16', False, True),
    ("LLaMA-3-8B + LoRA+ckpt", 8e9, 4096, 32, 'bf16', True,  True),
]

for name, params, hidden, layers, prec, ckpt, lora in scenarios:
    mb = memory_breakdown_gb(params, prec, hidden=hidden, layers=layers)
    total = mb['total']
    if lora:
        # LoRA: only adapter params in optimizer (1% of base), base frozen
        trainable_frac = 0.01
        total = (params * 2 / 1e9 +  # base model in bf16 (inference only, no grads)
                 params * trainable_frac * 2 / 1e9 +  # adapter params
                 params * trainable_frac * 2 / 1e9 +  # adapter grads
                 params * trainable_frac * 4 * 2 / 1e9 + # optimizer states for adapter
                 mb['activations'])
    if ckpt:
        total *= 0.6  # rough: checkpointing saves ~40% of activation memory

    fits = "\u2713 FITS" if total <= A10G_VRAM_GB else "\u2717 OOM"
    print(f"  {name:28s}  {prec}  {'ckpt' if ckpt else '    '}  "
          f"{'LoRA' if lora else '    '}  \u2192 {total:5.1f} GB  {fits}")

print()
gpt2_fp32_gb = memory_breakdown_gb(355e6, 'fp32', hidden=1024, layers=24)['total']
gpt2_bf16_gb = memory_breakdown_gb(355e6, 'bf16', hidden=1024, layers=24)['total']
llama1b_gb   = memory_breakdown_gb(1e9, 'bf16', hidden=2048, layers=22)['total'] * 0.6
llama8b_lora_gb = (8e9 * 2 / 1e9 + 8e9 * 0.01 * (2+2+8) / 1e9 +
                   memory_breakdown_gb(8e9,'bf16',hidden=4096,layers=32)['activations'] * 0.6)

fits_1b = "fits" if llama1b_gb <= A10G_VRAM_GB else "OOM"
fits_8b = "fits" if llama8b_lora_gb <= A10G_VRAM_GB else "OOM"

print(f"\nRiverside recommendation:")
print(f"  GPT-2-Medium at fp32:              {gpt2_fp32_gb:.1f} GB \u2713")
print(f"  GPT-2-Medium at bf16:              {gpt2_bf16_gb:.1f} GB \u2713")
print(f"  LLaMA-3-1B bf16+checkpointing:    {llama1b_gb:.1f} GB ({fits_1b})")
print(f"  LLaMA-3-8B bf16+LoRA+checkpointing: {llama8b_lora_gb:.1f} GB ({fits_8b})")

---

## Summary and Closing Decision

In [ ]:
# ── Closing Decision ──────────────────────────────────────────────────────────
print("=" * 60)
print("  CLOSING DECISION — Riverside A10G Model Selection")
print("=" * 60)
print()
print(f"  AWS A10G constraint: {A10G_VRAM_GB} GB VRAM")
print()
print(f"  GPT-2-Medium (fp32): {gpt2_fp32_gb:.1f} GB  \u2713  Full fine-tuning, no tricks needed")
print(f"  GPT-2-Medium (bf16): {gpt2_bf16_gb:.1f} GB  \u2713  Use torch.autocast for speed")
print(f"  LLaMA-3-1B (bf16+ckpt): {llama1b_gb:.1f} GB  {'\u2713' if llama1b_gb <= 24 else '\u2717'}  Need gradient checkpointing")
print(f"  LLaMA-3-8B (bf16+LoRA+ckpt): {llama8b_lora_gb:.1f} GB  {'\u2713' if llama8b_lora_gb <= 24 else '\u2717'}  Need LoRA + checkpointing")
print()
print("  KEY RULES:")
print("  1. Full fine-tuning costs ~4\u00d7 parameter memory (params + grads + optimizer)")
print("  2. Switch fp32\u2192bf16 to halve parameter + gradient memory")
print("  3. Add gradient checkpointing to cut activation memory by ~40% at +30% compute")
print("  4. LoRA reduces optimizer states from 4\u00d7 params to 4\u00d7 (1% of params) = 96% savings")
print()
print("  \u2192 For the 3-day sprint: start with GPT-2-Medium (guaranteed fit).")
print("  \u2192 For LLaMA-3-8B: LoRA + bf16 + checkpointing = the production recipe.")
print("    (See learning/genai/04-llm/02-llm-finetuning-parameter-techniques.ipynb for LoRA details)")

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated
- Memory footprint math — parameter × bytes formula for all precision formats
- fp16 overflow risk — demonstrated with or without GradScaler
- torch.autocast + GradScaler — stable mixed precision training
- Gradient checkpointing — memory vs. compute trade-off measured
- Memory profiling — peak VRAM measured across network depths
- Riverside model selection — concrete recommendation with computed numbers

### Tier 2 — Explained but Not Fully Built
- **CPU offloading** — ZeRO-Infinity moves optimizer states to CPU RAM; the memory math is shown but not implemented here

### Tier 3 — Named but Out of Scope
- **ZeRO-Offload / ZeRO-3** — DeepSpeed's extreme memory reduction through full parameter sharding; covered in Ch5 (Distributed Training)
- **Activation quantization** — quantise activations during forward pass to int8; more aggressive than checkpointing; covered in Ch6 (Quantization)

---

## When to Use What

| Memory pressure | Solution | Cost |
|---|---|---|
| Params barely fit | Switch fp32 → bf16 | None (may need GradScaler) |
| Activations too large | Gradient checkpointing | +30% compute per step |
| All weights too large | LoRA (freeze base, train adapters) | Slightly lower quality ceiling |
| Everything too large | LoRA + bf16 + checkpointing | Combined |
| Still doesn't fit | Multiple GPUs | See Ch5: Distributed Training |

→ **Next:** `learning/ai-infrastructure/05-distributed-training/` — when one GPU isn't enough.